# M10 — SMPL body through the static map (Parts A / A.5 / B)

Thin GPU runner for `tracking/smpl_person.py`. Logic lives in the fork; this notebook only clones, installs, mounts Drive, sets paths, runs, and displays.

- **Part A** — render BEHAVE **GT SMPL-H** moving through the reconstructed static surfel map.
- **Part A.5** — swap GT for **MAMMA-estimated SMPL-X** (run MAMMA on the 4 views first; §6).
- **Part B** — root-trajectory forecast, **ADE/FDE vs constant-velocity** (curve-walk sequence).

Fill in the Drive paths in §4, then Runtime ▸ Run all. Needs a GPU runtime (A100).

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2+3. Mount Drive + clone the fork + build CUDA submodules + deps
Same proven setup as the `behave_deform_run` notebooks, plus `smplx` (poses the body). The committed `render()` already accepts the means/rotation overrides `render_composite` uses — no runtime patch needed.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
if not os.path.isdir('/content/gaussian-surfel-local-map'):
    !git clone --recursive https://github.com/steptang/gaussian-surfel-local-map.git
sys.path.insert(0, '/content/gaussian-surfel-local-map')
os.chdir('/content/gaussian-surfel-local-map')
!git pull -q   # need the branch with tracking/smpl_person.py
!pip install -q plyfile open3d scikit-learn scipy mediapy imageio imageio-ffmpeg trimesh opencv-python-headless smplx
!pip install -q --no-build-isolation ./submodules/diff-surfel-rasterization
!pip install -q --no-build-isolation ./submodules/simple-knn
print('ready')

## 4. Paths  ⟵ EDIT THESE
- `CONV_ROOT` — converted per-timestep BEHAVE scenes (`timestep_*` dirs from the converter).
- `SMPL_ROOT` — raw BEHAVE sequence root (holds `t*.000/person/fit02/person_fit.pkl`).
- `SMPL_MODEL_DIR` — SMPL/SMPL-H/SMPL-X model files (`SMPLH_*.pkl`, `SMPLX_*.npz`).
- Pick a **walking-in-a-circle** sequence so Part B is meaningful.

In [ ]:
DRIVE = '/content/drive/MyDrive'
CONV_ROOT      = f'{DRIVE}/behave/converted/Date0X_Sub0Y_walk'   # TODO edit
SMPL_ROOT      = f'{DRIVE}/behave/raw/Date0X_Sub0Y_walk'         # TODO edit
SMPL_MODEL_DIR = f'{DRIVE}/body_models'                          # TODO edit
OUT_A   = '/content/out_partA_gt'
OUT_A5  = '/content/out_partA5_mamma'

## 5. Part A — GT SMPL body through the static map (+ Part B baseline)
First run: eyeball `compare.png`. If the body floats / is rotated / wrong scale, set `--align_t x y z` / `--align_s` and re-run (unknown (3)).

In [ ]:
!python -m tracking.smpl_person \
  --conv_root "$CONV_ROOT" --smpl_root "$SMPL_ROOT" --smpl_model_dir "$SMPL_MODEL_DIR" \
  --smpl_source gt --view 0 --out "$OUT_A" \
  --pred_hist 8 --pred_horizon 8

In [ ]:
import json
from IPython.display import Image, Video, display
OUT = OUT_A
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/prediction.json')), indent=2))
display(Image(f'{OUT}/compare.png'))
display(Video(f'{OUT}/sequence.mp4', embed=True, width=720))

## 6. Part A.5 — MAMMA as the SMPL source
1. Run MAMMA (`github.com/cuevhv/mamma`) on the 4 BEHAVE views per timestep → per-frame SMPL-X exports (one dir per frame, in the converted-timestep order).
2. Point `MAMMA_ROOT` at those exports and run below (`--smpl_source mamma`). Same downstream path; the only new bits are `load_mamma_smpl` (export layout — unknown (1)) and, if MAMMA's world frame differs, `--align_*` (unknown (3)).
3. Compare `person_psnr_mean`: GT vs MAMMA vs the deform-rig baseline.

In [ ]:
MAMMA_ROOT = f'{DRIVE}/behave/mamma_exports/Date0X_Sub0Y_walk'  # TODO edit (after running MAMMA)
# !python -m tracking.smpl_person \
#   --conv_root "$CONV_ROOT" --smpl_root "$MAMMA_ROOT" --smpl_model_dir "$SMPL_MODEL_DIR" \
#   --smpl_source mamma --view 0 --out "$OUT_A5" --pred_hist 8 --pred_horizon 8

## 7. Save outputs back to Drive

In [ ]:
SAVE_TO = f'{DRIVE}/behave/results/smpl_person'  # TODO edit
!mkdir -p "$SAVE_TO" && cp -r "$OUT_A" "$SAVE_TO/" && echo saved to "$SAVE_TO"